## Example 1: ELF Constants and Enumerations

In [ ]:
from elftools.elf.enums import *
import elftools

# Example 1: ELF constants and enumerations
print("=== ELF Constants and Enumerations ===")

# Library information
print(f"PyELFTools version: {elftools.__version__}")

# ELF file types
print(f"\nELF File Types:")
for name, value in list(ENUM_E_TYPE.items())[:6]:
    print(f"  {name}: {value}")

# Machine architectures  
print(f"\nMachine Architectures (common ones):")
common_archs = ['EM_NONE', 'EM_386', 'EM_X86_64', 'EM_ARM', 'EM_AARCH64', 'EM_MIPS']
for arch in common_archs:
    if arch in ENUM_E_MACHINE:
        print(f"  {arch}: {ENUM_E_MACHINE[arch]}")

# ELF identification values
print(f"\nELF Magic and Identification:")
print(f"Total machine types supported: {len(ENUM_E_MACHINE)}")
print(f"Total ELF file types: {len(ENUM_E_TYPE)}")
print(f"ELF versions: {dict(ENUM_E_VERSION)}")

# Dynamic section tags
print(f"\nDynamic Section Tags (sample):")
dt_tags = ['DT_NULL', 'DT_NEEDED', 'DT_PLTRELSZ', 'DT_PLTGOT', 'DT_HASH']
for tag in dt_tags:
    if tag in ENUM_D_TAG:
        print(f"  {tag}: {ENUM_D_TAG[tag]}")

print("✓ ELF constants and enumerations explored")

## Example 2: ELF Data Structures and Parsing Framework

In [ ]:
from elftools.elf.structs import ELFStructs
from elftools.elf.descriptions import *

# Example 2: ELF data structures and parsing framework
print("=== ELF Data Structures and Parsing Framework ===")

# Create ELF structures for different architectures
architectures = [
    ("32-bit little-endian", True, 32),
    ("64-bit little-endian", True, 64), 
    ("32-bit big-endian", False, 32),
    ("64-bit big-endian", False, 64)
]

elf_structures = {}
for desc, little_endian, elfclass in architectures:
    structs = ELFStructs(
        little_endian=little_endian,
        elfclass=elfclass
    )
    structs.create_basic_structs()  # the Elf_* definitions exist only after this (measured: none before)
    elf_structures[desc] = structs
    print(f"✓ Created {desc} ELF structures: {type(structs).__name__}")
    assert structs.Elf_Ehdr.sizeof() == {32: 52, 64: 64}[elfclass], structs.Elf_Ehdr.sizeof()  # the ELF header size the spec fixes

# Examine structure capabilities
print(f"\nELF Structure Analysis:")
if elf_structures:
    sample_structs = next(iter(elf_structures.values()))
    
    # Check available structure definitions
    struct_attrs = [attr for attr in dir(sample_structs) 
                   if not attr.startswith('_') and 'Elf_' in attr]
    
    print(f"Available ELF structure definitions: {len(struct_attrs)}")
    assert "Elf_Ehdr" in struct_attrs, struct_attrs
    for attr in struct_attrs[:8]:  # Show first 8
        print(f"  {attr}")

# Test description functions for various values
print(f"\nELF Field Descriptions:")

# describe_* take the ENUM NAME: an int is not a key and always comes back '<unknown>' (measured)
test_cases = [
    ("ELF classes", describe_ei_class, {"ELFCLASSNONE": "none", "ELFCLASS32": "ELF32", "ELFCLASS64": "ELF64"}),
    ("Data encodings", describe_ei_data, {"ELFDATANONE": "none", "ELFDATA2LSB": "2's complement, little endian",
                                          "ELFDATA2MSB": "2's complement, big endian"}),
    ("File types", describe_e_type, {"ET_NONE": "NONE (None)", "ET_REL": "REL (Relocatable file)", "ET_EXEC": "EXEC (Executable file)",
                                     "ET_DYN": "DYN (Shared object file)", "ET_CORE": "CORE (Core file)"}),
    ("Machine types", describe_e_machine, {"EM_NONE": "None", "EM_386": "Intel 80386", "EM_ARM": "ARM",
                                           "EM_X86_64": "Advanced Micro Devices X86-64", "EM_AARCH64": "AArch64"}),
]

for desc, func, expected in test_cases:
    print(f"\n{desc}:")
    for val, want in expected.items():
        result = func(val)
        print(f"  {val}: {result}")
        assert result == want, (desc, val, result)

print("✓ ELF structures and descriptions framework demonstrated")

## Example 3: DWARF Debug Information Support

In [ ]:
from elftools.dwarf.enums import *
from elftools.dwarf.structs import DWARFStructs
from elftools.dwarf.descriptions import *

# Example 3: DWARF debug information support
print("=== DWARF Debug Information Support ===")

# DWARF constants and enumerations
print("DWARF Constants:")
print(f"DWARF tags available: {len(ENUM_DW_TAG)}")
print(f"DWARF attributes available: {len(ENUM_DW_AT)}")
print(f"DWARF forms available: {len(ENUM_DW_FORM)}")

# Show common DWARF tags
print(f"\nCommon DWARF Tags:")
common_tags = [
    'DW_TAG_compile_unit', 'DW_TAG_subprogram', 'DW_TAG_variable',
    'DW_TAG_pointer_type', 'DW_TAG_base_type', 'DW_TAG_structure_type'
]
for tag in common_tags:
    if tag in ENUM_DW_TAG:
        print(f"  {tag}: {ENUM_DW_TAG[tag]}")

# Show common DWARF attributes
print(f"\nCommon DWARF Attributes:")
common_attrs = [
    'DW_AT_name', 'DW_AT_type', 'DW_AT_location', 
    'DW_AT_high_pc', 'DW_AT_low_pc', 'DW_AT_sibling'
]
for attr in common_attrs:
    if attr in ENUM_DW_AT:
        print(f"  {attr}: {ENUM_DW_AT[attr]}")

# Create DWARF structures for different formats
print(f"\nDWARF Structure Creation:")
dwarf_configs = [
    ("32-bit DWARF, little-endian", True, 32, 4),
    ("64-bit DWARF, little-endian", True, 64, 8),
    ("32-bit DWARF, big-endian", False, 32, 4)
]

for desc, little_endian, dwarf_format, address_size in dwarf_configs:
    dwarf_structs = DWARFStructs(
        little_endian=little_endian,
        dwarf_format=dwarf_format,
        address_size=address_size
    )
    print(f"✓ Created {desc}: {type(dwarf_structs).__name__}")
    assert (dwarf_structs.little_endian, dwarf_structs.dwarf_format, dwarf_structs.address_size) == (little_endian, dwarf_format, address_size)

# Show DWARF forms
print(f"\nDWARF Data Forms (sample):")
form_samples = ['DW_FORM_addr', 'DW_FORM_data1', 'DW_FORM_data4', 'DW_FORM_string']
for form in form_samples:
    if form in ENUM_DW_FORM:
        print(f"  {form}: {ENUM_DW_FORM[form]}")

print("✓ DWARF debug information framework demonstrated")

## Example 4: ELF Header Analysis and Binary Format Detection

In [ ]:
from elftools.elf.elffile import ELFFile
from elftools.common.exceptions import ELFError
import io
import tempfile
import os

# Example 4: ELF header analysis and binary format detection
print("=== ELF Header Analysis and Binary Format Detection ===")

def create_minimal_elf_header():
    """Create a minimal ELF header for demonstration"""
    # ELF magic number and basic header structure
    elf_header = bytearray(64)  # Minimum ELF header size
    
    # ELF magic: 0x7f + "ELF"
    elf_header[0:4] = b'\x7fELF'
    
    # EI_CLASS: 64-bit (2)
    elf_header[4] = 2  
    
    # EI_DATA: Little-endian (1)
    elf_header[5] = 1
    
    # EI_VERSION: Current version (1)  
    elf_header[6] = 1
    
    # EI_OSABI: System V ABI (0)
    elf_header[7] = 0
    
    # Remaining identification bytes (padding)
    elf_header[8:16] = b'\x00' * 8
    
    # e_type: Executable file (2) - little-endian 2-byte
    elf_header[16:18] = b'\x02\x00'
    
    # e_machine: x86-64 (62) - little-endian 2-byte  
    elf_header[18:20] = b'\x3e\x00'
    
    # e_version: Current version (1) - little-endian 4-byte
    elf_header[20:24] = b'\x01\x00\x00\x00'
    
    return bytes(elf_header)

# Create and analyze ELF header
print("ELF Header Creation and Analysis:")

try:
    # Create minimal ELF header
    elf_data = create_minimal_elf_header()
    print(f"✓ Created minimal ELF header: {len(elf_data)} bytes")
    
    # Show ELF magic verification
    magic = elf_data[:4]
    print(f"ELF magic bytes: {magic} ({'Valid ELF' if magic == b'\\x7fELF' else 'Invalid'})")
    
    # Parse identification fields manually
    ei_class = elf_data[4]
    ei_data = elf_data[5] 
    ei_version = elf_data[6]
    ei_osabi = elf_data[7]
    
    print(f"EI_CLASS: {ei_class} ({'64-bit' if ei_class == 2 else '32-bit' if ei_class == 1 else 'Unknown'})")
    print(f"EI_DATA: {ei_data} ({'Little-endian' if ei_data == 1 else 'Big-endian' if ei_data == 2 else 'Unknown'})")
    print(f"EI_VERSION: {ei_version}")
    print(f"EI_OSABI: {ei_osabi}")
    
    # Parse file type and machine
    import struct
    e_type = struct.unpack('<H', elf_data[16:18])[0]  # Little-endian short
    e_machine = struct.unpack('<H', elf_data[18:20])[0]
    e_version = struct.unpack('<I', elf_data[20:24])[0]  # Little-endian int
    
    print(f"e_type: {e_type} ({'Executable' if e_type == 2 else 'Other'})")
    print(f"e_machine: {e_machine} ({'x86-64' if e_machine == 62 else 'Other'})")
    print(f"e_version: {e_version}")
    
except Exception as e:
    print(f"✗ Error creating ELF header: {e}")
    raise

# Demonstrate binary format detection
print(f"\nBinary Format Detection:")

def detect_binary_format(data):
    """Simple binary format detection"""
    if len(data) < 4:
        return "Too short to identify"
    
    magic = data[:4]
    if magic == b'\x7fELF':
        return "ELF (Executable and Linkable Format)"
    elif magic[:2] == b'MZ':
        return "PE (Portable Executable - Windows)"
    elif magic == b'\xca\xfe\xba\xbe' or magic == b'\xfe\xed\xfa\xce':
        return "Mach-O (macOS/iOS executable)"
    elif magic[:2] == b'\x4d\x5a':
        return "DOS executable"
    else:
        return f"Unknown format (magic: {magic.hex()})"

# Test format detection
test_formats = [
    (b'\x7fELF\x02\x01\x01\x00', "ELF header"),
    (b'MZ\x90\x00', "PE/DOS header"), 
    (b'\xca\xfe\xba\xbe', "Mach-O header"),
    (b'\x50\x4b\x03\x04', "ZIP header"),
]

for data, desc in test_formats:
    format_detected = detect_binary_format(data)
    print(f"  {desc}: {format_detected}")

print("✓ ELF header analysis and format detection demonstrated")

## Example 5: Advanced ELF Analysis - Sections, Segments, and Symbols

In [ ]:
from elftools.elf.sections import *
from elftools.elf.segments import *
from elftools.elf.enums import ENUM_E_TYPE, ENUM_E_MACHINE, ENUM_D_TAG

# Example 5: Advanced ELF analysis - sections, segments, and symbols
print("=== Advanced ELF Analysis Framework ===")

# ELF file types analysis
print("ELF File Types Analysis:")
file_types = {
    'ET_NONE': 'No file type', 'ET_REL': 'Relocatable file',
    'ET_EXEC': 'Executable file', 'ET_DYN': 'Shared object file',
    'ET_CORE': 'Core file'
}

for type_name, description in file_types.items():
    if type_name in ENUM_E_TYPE:
        value = ENUM_E_TYPE[type_name]
        print(f"  {type_name} ({value}): {description}")

# Dynamic section tags
print(f"\nDynamic Section Tags (sample):")
dynamic_tags = ['DT_NULL', 'DT_NEEDED', 'DT_STRTAB', 'DT_SYMTAB', 'DT_SONAME']

for tag in dynamic_tags:
    if tag in ENUM_D_TAG:
        print(f"  {tag}: {ENUM_D_TAG[tag]}")

# Architecture support summary
print(f"\nFramework Capabilities:")
print(f"  Total file types: {len(ENUM_E_TYPE)}")
print(f"  Total architectures: {len(ENUM_E_MACHINE)}")
print(f"  Total dynamic tags: {len(ENUM_D_TAG)}")
print("✓ Advanced ELF analysis framework demonstrated")